# 00 - Load the event data and check its quality

**Goal.** Make the 68 Parquet files queryable with DuckDB, and check what is in them before any analysis.

**Approach.** The files are not copied into a database (3.3 GB): DuckDB reads Parquet directly, and a *view*
named `events` is created over the readable files. The view lives only for this session; each notebook
re-creates it with the same helper (`scripts/load_events.py`).

In [1]:
import glob
import sys
sys.path.append("../scripts")          # to import the helper below
from load_events import connect_events

con, readable, unreadable = connect_events("../data/raw/player_analytics/*.parquet")
print(len(readable), "readable files,", len(unreadable), "unreadable")
for f in unreadable:
    print("skipped:", f)

67 readable files, 1 unreadable
skipped: ../data/raw/player_analytics\part-00049-a228a717-fd60-4fea-83bd-edbe775a63d0.c000.zstd.parquet


## Schema

Each row is one event. `event_properties` and `user_properties_custom` are key-value maps (flexible, game-specific details).
Timestamps are stored as text and cast to `TIMESTAMP` in the view.

In [2]:
con.sql("DESCRIBE events").show(max_width=200)
con.sql("SELECT * FROM events LIMIT 3").show(max_width=250)

┌────────────────────────┬───────────────────────┬─────────┬─────────┬─────────┬─────────┐
│      column_name       │      column_type      │  null   │   key   │ default │  extra  │
│        varchar         │        varchar        │ varchar │ varchar │ varchar │ varchar │
├────────────────────────┼───────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ event_id               │ VARCHAR               │ YES     │ NULL    │ NULL    │ NULL    │
│ event_name             │ VARCHAR               │ YES     │ NULL    │ NULL    │ NULL    │
│ event_properties       │ MAP(VARCHAR, VARCHAR) │ YES     │ NULL    │ NULL    │ NULL    │
│ event_receive_time     │ VARCHAR               │ YES     │ NULL    │ NULL    │ NULL    │
│ user_id                │ VARCHAR               │ YES     │ NULL    │ NULL    │ NULL    │
│ user_session_id        │ VARCHAR               │ YES     │ NULL    │ NULL    │ NULL    │
│ user_session_issued_at │ VARCHAR               │ YES     │ NULL    │ NULL    │ NULL    │

## Size of the dataset

In [3]:
con.sql("""
    SELECT COUNT(*)                        AS events,
           COUNT(DISTINCT user_id)         AS players,
           COUNT(DISTINCT user_session_id) AS sessions,
           MIN(ts)                         AS first_event,
           MAX(ts)                         AS last_event
    FROM events
""").show(max_width=200)

┌──────────┬─────────┬──────────┬─────────────────────┬─────────────────────┐
│  events  │ players │ sessions │     first_event     │     last_event      │
│  int64   │  int64  │  int64   │      timestamp      │      timestamp      │
├──────────┼─────────┼──────────┼─────────────────────┼─────────────────────┤
│ 70931231 │   71679 │   275698 │ 2025-02-28 05:58:43 │ 2025-04-10 20:37:24 │
└──────────┴─────────┴──────────┴─────────────────────┴─────────────────────┘



## Quality checks

1. Duplicated `event_id` (the same event received twice), checked on one day.
2. Missing values in the key columns.
3. Daily active players: a sudden drop points to missing data.

In [4]:
# 1. Duplicates: an event received twice would inflate every metric.
# An exact check on all 71 million ids needs a lot of memory, so it is done on one ordinary day.
con.sql("""
    SELECT COUNT(*)                 AS events,
           COUNT(DISTINCT event_id) AS distinct_event_ids
    FROM events
    WHERE ts::DATE = DATE '2025-03-15'
""").show()

┌─────────┬────────────────────┐
│ events  │ distinct_event_ids │
│  int64  │       int64        │
├─────────┼────────────────────┤
│ 1489753 │            1489753 │
└─────────┴────────────────────┘



In [5]:
# 2. Missing values in the columns used later.
con.sql("""
    SELECT
      COUNT(*) FILTER (WHERE user_id IS NULL)         AS null_user,
      COUNT(*) FILTER (WHERE user_session_id IS NULL) AS null_session,
      COUNT(*) FILTER (WHERE ts IS NULL)              AS null_ts,
      COUNT(*) FILTER (WHERE client_platform = '' OR client_platform IS NULL) AS empty_platform
    FROM events
""").show()

┌───────────┬──────────────┬─────────┬────────────────┐
│ null_user │ null_session │ null_ts │ empty_platform │
│   int64   │    int64     │  int64  │     int64      │
├───────────┼──────────────┼─────────┼────────────────┤
│         0 │            0 │       0 │          31977 │
└───────────┴──────────────┴─────────┴────────────────┘



In [6]:
# 3. Daily active players. The dip on 2025-03-27 is explained in data/README.md.
con.sql("""
    SELECT ts::DATE AS day, COUNT(DISTINCT user_id) AS daily_active_players
    FROM events
    GROUP BY day
    ORDER BY day
""").show(max_rows=50)

┌────────────┬──────────────────────┐
│    day     │ daily_active_players │
│    date    │        int64         │
├────────────┼──────────────────────┤
│ 2025-02-28 │                 2622 │
│ 2025-03-01 │                 5001 │
│ 2025-03-02 │                 4814 │
│ 2025-03-03 │                 4108 │
│ 2025-03-04 │                 4541 │
│ 2025-03-05 │                 4567 │
│ 2025-03-06 │                 4715 │
│ 2025-03-07 │                 4869 │
│ 2025-03-08 │                 4999 │
│ 2025-03-09 │                 4568 │
│ 2025-03-10 │                 3622 │
│ 2025-03-11 │                 3425 │
│ 2025-03-12 │                 3214 │
│ 2025-03-13 │                 3301 │
│ 2025-03-14 │                 3507 │
│ 2025-03-15 │                 3940 │
│ 2025-03-16 │                 3961 │
│ 2025-03-17 │                 3663 │
│ 2025-03-18 │                 3501 │
│ 2025-03-19 │                 3534 │
│ 2025-03-20 │                 3408 │
│ 2025-03-21 │                 3276 │
│ 2025-03-22

In [7]:
con.sql("DESCRIBE events").show(max_rows=100)
con.sql("SELECT * FROM events LIMIT 5").show(max_width=200)

┌────────────────────────┬───────────────────────┬─────────┬─────────┬─────────┬─────────┐
│      column_name       │      column_type      │  null   │   key   │ default │  extra  │
│        varchar         │        varchar        │ varchar │ varchar │ varchar │ varchar │
├────────────────────────┼───────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ event_id               │ VARCHAR               │ YES     │ NULL    │ NULL    │ NULL    │
│ event_name             │ VARCHAR               │ YES     │ NULL    │ NULL    │ NULL    │
│ event_properties       │ MAP(VARCHAR, VARCHAR) │ YES     │ NULL    │ NULL    │ NULL    │
│ event_receive_time     │ VARCHAR               │ YES     │ NULL    │ NULL    │ NULL    │
│ user_id                │ VARCHAR               │ YES     │ NULL    │ NULL    │ NULL    │
│ user_session_id        │ VARCHAR               │ YES     │ NULL    │ NULL    │ NULL    │
│ user_session_issued_at │ VARCHAR               │ YES     │ NULL    │ NULL    │ NULL    │

In [8]:
con.sql("""
SELECT event_name, k AS property, COUNT(*) AS n
FROM (SELECT event_name, UNNEST(map_keys(event_properties)) AS k
      FROM events USING SAMPLE 1 PERCENT)
GROUP BY 1, 2 ORDER BY 1, n DESC
""").show(max_rows=200)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────────┬──────────────────────────┬───────┐
│       event_name       │         property         │   n   │
│        varchar         │         varchar          │ int64 │
├────────────────────────┼──────────────────────────┼───────┤
│ _experimentJoin        │ experiment_phase_id      │    40 │
│ _experimentJoin        │ experiment_phase_name    │    40 │
│ _experimentJoin        │ experiment_variant_id    │    40 │
│ _experimentJoin        │ experiment_name          │    40 │
│ _experimentJoin        │ experiment_id            │    40 │
│ _experimentJoin        │ experiment_variant_name  │    40 │
│ _identityCreate        │ identity_id              │   466 │
│ _sessionStart          │ session_origin           │  2803 │
│ _sessionStart          │ token_expires_at         │  2803 │
│ _sessionStart          │ session_id               │  2803 │
│ _sessionStart          │ identity_id              │  2803 │
│ _sessionStart          │ refresh_token_expires_at │  2583 │
│ achiev

In [9]:
#con.close()